In [1]:
from util.stream import Stream, transform_stream
from util.drift_generator import DriftGenerator, TimeSeriesDrifter
import os
import yaml

### Load data: anomaly injected three weather dataset

In [2]:
# Parse source filepaths from config (.yaml) file
with open('Weather_test.yaml') as f:
    config_param = yaml.load(f, Loader=yaml.FullLoader)
source_dir = os.path.abspath(config_param['source_dir'])
GR_filepath = f"{source_dir}/{config_param['source_files'][0]}"
NO_filepath = f"{source_dir}/{config_param['source_files'][1]}"
DE_filepath = f"{source_dir}/{config_param['source_files'][2]}"

GR = Stream(GR_filepath)
NO = Stream(NO_filepath)
DE = Stream(DE_filepath)


WeatherStreams = []
WeatherStreams.append(GR)
WeatherStreams.append(NO)
WeatherStreams.append(DE)

### 1. Inject Drift
- Varying n_drift

In [ ]:
moa_path = config_param['moa_path']
source_streams = WeatherStreams
drift_dir = os.getcwd()+ config_param['drift_dir']
g = DriftGenerator(source_dir, drift_dir, moa_path, selected_streams=source_streams)

length = WeatherStreams[0].length
dataset = 'WeatherTest_n_d'

n_drift = config_param['drift_params']['n_drift']
p_drift = config_param['drift_params']['p_drift']
p_before = config_param['drift_params']['p_before']
sub_dir = config_param['drift_params']['sub_dir']

for n_d in n_drift:
    ds = g.run_generate_grad_stream_moa(
        length=length,
        p_drift = 0.2,
        n_drift = n_d,
        p_before = p_before,
        sub_dir=sub_dir,
        dataset = dataset,
        mode=0        
    )

    # ds.plot_drift()

### Varying p_drift
- p_drift

In [ ]:
dataset = 'WeatherTest_p_d'
for p_d in p_drift:
    ds = g.run_generate_grad_stream_moa(
        length=length,
        p_drift = p_d,
        n_drift = 5,
        p_before = p_before,
        sub_dir=sub_dir,
        dataset = dataset,
        mode=0        
    )

    # ds.plot_drift()

### Modulate data

In [ ]:
streams = [GR, NO]
varying_v = [0.5, 1, 2]
varying_o = [0.1, 0.5, 1, 3]
varying_m = [0, 0.5, 1, 1.5]
varying_p = [0.8, 1, 1.2]


for s in streams:
    td = TimeSeriesDrifter(100)
    td.modulate_stream(s, varying_v[0], varying_o[1], varying_m[1])

    td = TimeSeriesDrifter(100)
    td.modulate_stream(s, varying_v[2], varying_o[3], varying_m[2])

    td = TimeSeriesDrifter(100)
    td.modulate_stream(s, varying_v[1], varying_o[0], varying_m[3])

In [6]:
### Need to select 'source_files' here
## How many? --> three? (similar to original?)
## which ones to choose? (v, o, m)-> 3-pairs
s = 'GR'
v_s=[0.5, 2, 1]
o_s=[0.5, 3, 0.1]
m_s=[0.5, 1, 1.5]
selected_files = []
f_name = f'{s}_temperature_anomaly.arff'
selected_files.append(f_name)
for v, o, m in zip(v_s, o_s, m_s):
    f_name = f'{s}_temperature_anomaly_v{v}_o{o}_m{m}_p1_mod.arff'
    selected_files.append(f_name)

# selected_files = config_param['source_files']

dir = config_param['source_dir']
WeatherStreams_mod = []
for i, file in enumerate(selected_files):
    WeatherStreams_mod.append(Stream(f"{dir}/{file}"))
    print(WeatherStreams_mod[i].filename)


GR_temperature_anomaly
GR_temperature_anomaly_v0.5_o0.5_m0.5_p1_mod
GR_temperature_anomaly_v2_o3_m1_p1_mod
GR_temperature_anomaly_v1_o0.1_m1.5_p1_mod


In [ ]:
moa_path = config_param['moa_path']
source_streams = WeatherStreams_mod
drift_dir = os.getcwd()+ config_param['drift_dir']
g = DriftGenerator(dir, drift_dir, moa_path, selected_streams=source_streams)

length = WeatherStreams_mod[0].length
dataset = f'{s}Test_n_d'

n_drift = config_param['drift_params']['n_drift']
p_drift = config_param['drift_params']['p_drift']
p_before = config_param['drift_params']['p_before']
# sub_dir = config_param['drift_params']['sub_dir']
sub_dir = 'test_mod_weather'

for n_d in n_drift:
    ds = g.run_generate_grad_stream_moa(
        length=length,
        p_drift = 0.2,
        n_drift = n_d,
        p_before = p_before,
        sub_dir=sub_dir,
        dataset = dataset,
        mode=0        
    )

    ds.plot_drift()

In [ ]:
dataset = f'{s}Test_p_d'
for p_d in p_drift:
    ds = g.run_generate_grad_stream_moa(
        length=length,
        p_drift = p_d,
        n_drift = 5,
        p_before = p_before,
        sub_dir=sub_dir,
        dataset = dataset,
        mode=0        
    )

    ds.plot_drift()